<div style="padding:28px 32px;border-radius:16px;background:linear-gradient(135deg,#0B1F33,#123C69);color:white">
  <div style="font-size:14px;letter-spacing:.08em;text-transform:uppercase;color:#86D7FF">Python programavimas finansinėms technologijoms</div>
  <h1 style="margin:.35em 0 .2em 0;color:white">Python kitaip</h1>
  <div style="font-size:22px">Pythoniškos konstrukcijos ir standartinė biblioteka FinTech uždaviniams</div>
</div>

**Paskirtis.** Ši užrašinė skirta studentui, kuris jau matė kintamuosius, sąlygas, ciklus ir funkcijas, tačiau dar nepažįsta Python idiomų. Pavyzdžiai rodo ne triukus, o būdus aiškiau išreikšti duomenų apdorojimo ketinimą.

> **Svarbi formuluotė.** Dalis priemonių nėra absoliučiai unikalios Python kalbai. Jos išskirtinės tuo, kad yra glaudžiai integruotos į kalbos sintaksę, duomenų modelį ir standartinę biblioteką, todėl dažnai sudaro atpažįstamą „pythonišką“ sprendimo stilių.

Visi duomenys yra maži, sintetiniai ir įrašyti pačioje užrašinėje. Kursų, rizikos slenksčių ir verslo taisyklių reikšmės yra **pedagoginės**, ne rinkos duomenys ir ne finansinė rekomendacija.


## Mokymosi rezultatai

Baigęs šią užrašinę studentas gebės:

1. pakeisti indeksais paremtus ciklus tiesiogine iteracija, `enumerate()` ir `zip(strict=True)`;
2. taikyti išpakavimą, pjūvius, grandininius palyginimus ir comprehensions nepakenkdamas skaitomumui;
3. apdoroti duomenų srautą generatoriais ir `itertools` priemonėmis;
4. pinigų sumoms pasirinkti `Decimal`, o laiko momentams — laiko juostą turintį `datetime`;
5. struktūruoti domeno duomenis su `dataclass` ir saugiai valdyti išteklius su `with`;
6. atpažinti, kada trumpesnis Python kodas tampa pernelyg gudrus.

**Suderinamumas:** pavyzdžiams reikia Python 3.10 ar naujesnės versijos; išorinės bibliotekos nenaudojamos.


## 0. Aplinka ir atkuriami pradiniai duomenys

FinTech analizėje pirmiausia turime žinoti, kokioje aplinkoje gautas rezultatas. Toliau patikriname versiją ir sukuriame mažą nekintantį duomenų rinkinį. Sumos kol kas saugomos tekstu: taip vėliau galėsime jas tiksliai paversti į `Decimal`.


In [3]:
import sys
from datetime import datetime
from decimal import Decimal

assert sys.version_info >= (3, 10), "Šiai užrašinei reikia Python 3.10+."
print("Python versija:", sys.version.split()[0])

sandoriai = [
    {"id": "TX-001", "klientas": "C101", "suma": "125.40", "valiuta": "EUR", "busena": "ivykdytas", "rizikos_balas": 0.12},
    {"id": "TX-002", "klientas": "C205", "suma": "980.00", "valiuta": "USD", "busena": "ivykdytas", "rizikos_balas": 0.81},
    {"id": "TX-003", "klientas": "C101", "suma": "45.10",  "valiuta": "EUR", "busena": "atmestas",  "rizikos_balas": 0.66},
    {"id": "TX-004", "klientas": "C330", "suma": "2500.00","valiuta": "GBP", "busena": "ivykdytas", "rizikos_balas": 0.93},
    {"id": "TX-005", "klientas": "C205", "suma": "310.25", "valiuta": "EUR", "busena": "ivykdytas", "rizikos_balas": 0.37},
]

datos = ["2026-09-07", "2026-09-08", "2026-09-09", "2026-09-10", "2026-09-11"]
uzdarymo_kainos = [100.00, 101.50, 100.80, 103.20, 102.70]

len(sandoriai), len(uzdarymo_kainos)
help(len)


Python versija: 3.13.7
Help on built-in function len in module builtins:

len(obj, /)
    Return the number of items in a container.



---

## 1. Iteruojame per objektus, ne per indeksus

Kitose kalbose dažnas įprotis — kurti skaitiklį ir juo pasiekti masyvo elementą. Python seka jau yra iteruojamas objektas, todėl indeksas neturėtų būti įtraukiamas be reikalo.

**FinTech pritaikymas:** transakcijų validavimas, pavedimų būsenų tikrinimas, kainų srauto apdorojimas. Tiesioginė iteracija sumažina klaidos „vienu indeksu per daug“ tikimybę.


In [4]:
# Veikia, bet indeksas čia neturi verslo prasmės.
ids_per_indeksa = []
for i in range(len(sandoriai)):
    ids_per_indeksa.append(sandoriai[i]["id"])

# Pythoniškas variantas: vardas nusako patį objektą.
ids_tiesiogiai = []
for sandoris in sandoriai:
    ids_tiesiogiai.append(sandoris["id"])

assert ids_per_indeksa == ids_tiesiogiai
ids_tiesiogiai


['TX-001', 'TX-002', 'TX-003', 'TX-004', 'TX-005']

Kai reikia ir eilės numerio, naudojame `enumerate()`. Parametras `start=1` leidžia rodyti žmogui įprastą numeraciją nekeičiant duomenų indeksų.


In [5]:
for eil_nr, sandoris in enumerate(sandoriai, start=1):
    if sandoris["rizikos_balas"] >= 0.80:
        print(f"{eil_nr}. peržiūrėti {sandoris['id']} (balas {sandoris['rizikos_balas']:.2f})")


2. peržiūrėti TX-002 (balas 0.81)
4. peržiūrėti TX-004 (balas 0.93)


### Lygiagrečios sekos: `zip(..., strict=True)`

`zip()` poruoja kelių sekų elementus. Finansiniuose duomenyse tylus trumpesnės sekos nukirpimas gali paslėpti trūkstamą kainą ar datą, todėl Python 3.10+ verta įjungti `strict=True`: nesutampant ilgiams gaunama aiški klaida.


In [6]:
kainos_pagal_data = dict(zip(datos, uzdarymo_kainos, strict=True))
print(kainos_pagal_data)

try:
    list(zip(datos, uzdarymo_kainos[:-1], strict=True))
except ValueError as klaida:
    print("Aptiktas ilgių neatitikimas:", klaida)


{'2026-09-07': 100.0, '2026-09-08': 101.5, '2026-09-09': 100.8, '2026-09-10': 103.2, '2026-09-11': 102.7}
Aptiktas ilgių neatitikimas: zip() argument 2 is shorter than argument 1


> **Kada netinka?** Jeigu trumpesnės sekos nukirpimas yra sąmoningas algoritmo sprendimas, paprastas `zip()` gali būti tinkamas, bet šią prielaidą reikia užrašyti. Finansinių duomenų derinime saugesnė numatytoji nuostata dažniausiai yra `strict=True`.


---

## 2. Išpakavimas: duomenų struktūra matoma kairėje

Python leidžia vienu priskyrimu išpakuoti seką į prasmingai pavadintus kintamuosius. Žvaigždute pažymėtas vardas surenka likusius elementus.

**FinTech pritaikymas:** pirmos ir paskutinės kainos atskyrimas, CSV eilutės laukų paėmimas, geriausio ir likusių pasiūlymų nagrinėjimas.


In [7]:
pirma_kaina, *tarpines_kainos, paskutine_kaina = uzdarymo_kainos
pokytis = paskutine_kaina / pirma_kaina - 1

print("Pirma kaina:", pirma_kaina)
print("Tarpinių kainų skaičius:", len(tarpines_kainos))
print("Paskutinė kaina:", paskutine_kaina)
print(f"Laikotarpio pokytis: {pokytis:.2%}")


Pirma kaina: 100.0
Tarpinių kainų skaičius: 3
Paskutinė kaina: 102.7
Laikotarpio pokytis: 2.70%


In [8]:
# Poros išpakuojamos tiesiog ciklo antraštėje.
valiutu_kursai = [("USD", Decimal("0.92")), ("GBP", Decimal("1.17"))]
for valiuta, eur_uz_vieneta in valiutu_kursai:
    print(f"1 {valiuta} = {eur_uz_vieneta} EUR (iliustracinis kursas)")

# Apsikeitimui nereikia laikino kintamojo.
pirkimo_kaina, pardavimo_kaina = Decimal("101.20"), Decimal("101.35")
pirkimo_kaina, pardavimo_kaina = pardavimo_kaina, pirkimo_kaina
assert pirkimo_kaina == Decimal("101.35")


1 USD = 0.92 EUR (iliustracinis kursas)
1 GBP = 1.17 EUR (iliustracinis kursas)


> **Riba:** išpakavimas naudingas, kai struktūra stabili ir aiški. Jei CSV stulpelių skaičius gali keistis, pirmiau tikrinkite antraštes arba naudokite `csv.DictReader`, o ne remkitės vien pozicija.


---

## 3. Neigiami indeksai ir pjūviai

`seka[-1]` reiškia paskutinį elementą, o `seka[a:b:c]` — pjūvį nuo `a` iki `b` (neįtraukiant `b`) žingsniu `c`.

**FinTech pritaikymas:** paskutinė rinkos kaina, slenkantis istorinis langas, naujausi pavedimai, chronologinės sekos apvertimas diagnostikai.


In [10]:
paskutine = uzdarymo_kainos[-1]
paskutines_trys = uzdarymo_kainos[-3:]
chronologija_atbulai = uzdarymo_kainos[::-1]

print("Paskutinė kaina:", paskutine)
print("3 paskutinės:", paskutines_trys)
print("Atbulai:", chronologija_atbulai)


Paskutinė kaina: 102.7
3 paskutinės: [100.8, 103.2, 102.7]
Atbulai: [102.7, 103.2, 100.8, 101.5, 100.0]


### Chronologinis atskyrimas be ateities informacijos

Laiko eilutėje atsitiktinis sumaišymas gali sukelti **perspektyvinės informacijos nutekėjimą**. Pjūviu patogu išsaugoti tvarką: pradžia skiriama mokymui, pabaiga — tikrinimui.


In [11]:
riba = int(len(uzdarymo_kainos) * 0.60)
mokymo_kainos = uzdarymo_kainos[:riba]
tikrinimo_kainos = uzdarymo_kainos[riba:]

assert mokymo_kainos[-1] == uzdarymo_kainos[riba - 1]
assert tikrinimo_kainos[0] == uzdarymo_kainos[riba]
print("Mokymas:", mokymo_kainos)
print("Tikrinimas:", tikrinimo_kainos)


Mokymas: [100.0, 101.5, 100.8]
Tikrinimas: [103.2, 102.7]


> **Metodologinė pastaba:** vien chronologinio pjūvio nepakanka rimtam modelio vertinimui. Reikia pagrįsti ribą, kartais taikyti slenkantį vertinimą, įtraukti transakcijų sąnaudas ir užtikrinti, kad visi požymiai būtų prieinami prognozės momentu.


---

## 4. Grandininiai palyginimai, narystė ir „truthiness“

Matematinę sąlygą Python galima rašyti tiesiogiai: `0 <= x < 1`. Ji ne tik trumpesnė už dvi su `and` sujungtas sąlygas, bet ir aiškiau perteikia intervalo idėją.


In [13]:
balai_tinkami = all(0.0 <= s["rizikos_balas"] <= 1.0 for s in sandoriai)
leistinos_busenos = {"ivykdytas", "atmestas", "laukiama"}
busenos_tinkamos = all(s["busena"] in leistinos_busenos for s in sandoriai)

print("Rizikos balai [0, 1]:", balai_tinkami)
print("Būsenos iš žodyno:", busenos_tinkamos)
assert balai_tinkami and busenos_tinkamos


Rizikos balai [0, 1]: True
Būsenos iš žodyno: True


Tuščia seka, nulis ir `None` loginėje išraiškoje laikomi klaidingais, tačiau finansuose jų prasmės skiriasi:

- `None` gali reikšti „duomens nėra“;
- `Decimal("0")` — tikrą nulinę sumą;
- `[]` — kad nerasta nė vieno įrašo.

Todėl `if not suma:` gali netyčia suplakti trūkstamą ir nulinę reikšmę.


In [14]:
def sumos_busena(suma):
    if suma is None:
        return "trūksta"
    if suma == 0:
        return "nulinė"
    return "yra reikšmė"

rezultatai = [sumos_busena(None), sumos_busena(Decimal("0")), sumos_busena(Decimal("12.50"))]
print(rezultatai)
assert rezultatai == ["trūksta", "nulinė", "yra reikšmė"]


['trūksta', 'nulinė', 'yra reikšmė']


> **Žvilgsnis į priekį:** kai vėliau naudosite lentelių ar masyvų bibliotekas, viso masyvo loginė reikšmė gali būti dviprasmė. Sąlygas reikės redukuoti aiškiai, pavyzdžiui, su `any()` arba `all()`.


---

## 5. Comprehensions: transformacija ir filtras vienoje išraiškoje

List, set ir dict comprehensions tinka trumpai, vieno žingsnio transformacijai. Skaitome jas tokia tvarka: **ką kuriame → iš ko → su kokia sąlyga**.

**FinTech pritaikymas:** rizikingų ID atranka, unikalių valiutų žodynas, duomenų normalizavimas prieš tolesnę analizę.


In [15]:
rizikingi_id = [
    s["id"]
    for s in sandoriai
    if s["rizikos_balas"] >= 0.80
]

valiutos = {s["valiuta"] for s in sandoriai}
balas_pagal_id = {s["id"]: s["rizikos_balas"] for s in sandoriai}

print("Rizikingi:", rizikingi_id)
print("Valiutos:", sorted(valiutos))
print("TX-004 balas:", balas_pagal_id["TX-004"])


Rizikingi: ['TX-002', 'TX-004']
Valiutos: ['EUR', 'GBP', 'USD']
TX-004 balas: 0.93


> **Kada sustoti?** Jei išraiškai reikia kelių įdėtų ciklų, kelių sąlygų, šalutinių efektų ar ilgo verslo taisyklės paaiškinimo, įprastas ciklas arba vardinė funkcija bus aiškesni. Trumpumas nėra savarankiškas kokybės kriterijus.


---

## 6. Integruotos funkcijos kaip mažas duomenų apdorojimo žodynas

`any`, `all`, `sum`, `min`, `max` ir `sorted` dažnai pakeičia pagalbinius ciklus. Parametras `key=` leidžia perduoti kriterijų kaip funkciją — tai pirmos klasės funkcijų pavyzdys.

**FinTech pritaikymas:** kontrolinių taisyklių rinkinys, didžiausia ekspozicija, prioritetinė rizikos peržiūros eilė.


In [16]:
ar_yra_didele_rizika = any(s["rizikos_balas"] >= 0.90 for s in sandoriai)
didziausios_rizikos = max(sandoriai, key=lambda s: s["rizikos_balas"])
pagal_rizika = sorted(sandoriai, key=lambda s: s["rizikos_balas"], reverse=True)

print("Yra ≥ 0.90 rizika:", ar_yra_didele_rizika)
print("Didžiausios rizikos ID:", didziausios_rizikos["id"])
print("Peržiūros eilė:", [s["id"] for s in pagal_rizika])


Yra ≥ 0.90 rizika: True
Didžiausios rizikos ID: TX-004
Peržiūros eilė: ['TX-004', 'TX-002', 'TX-003', 'TX-005', 'TX-001']


`lambda` čia tinka, nes kriterijus trumpas. Jei kriterijus įgyvendina svarbią verslo taisyklę, geriau suteikti jam vardą — jį bus lengviau testuoti ir paaiškinti auditui.


In [17]:
def perziuros_prioritetas(sandoris):
    """Didesnis rizikos balas ir suma reiškia ankstesnę peržiūrą."""
    return (sandoris["rizikos_balas"], Decimal(sandoris["suma"]))

prioritetine_eile = sorted(sandoriai, key=perziuros_prioritetas, reverse=True)
print([s["id"] for s in prioritetine_eile])


['TX-004', 'TX-002', 'TX-003', 'TX-005', 'TX-001']


---

## 7. Generatoriai: skaičiuojame pareikalavus

Sąrašo comprehension iš karto sukuria visus rezultatus. Generatorinė išraiška naudoja skliaustus ir reikšmes pateikia po vieną. Funkcija su `yield` pati tampa generatoriaus gamykla.

**FinTech pritaikymas:** dideli operacijų žurnalai, rinkos pranešimų srautas, filtravimo grandinė, kur visų tarpinių reikšmių laikyti atmintyje nereikia.


In [18]:
# Reikšmės konvertuojamos tik tada, kai sum() jų paprašo.
ivykdytu_eur_sumos = (
    Decimal(s["suma"])
    for s in sandoriai
    if s["busena"] == "ivykdytas" and s["valiuta"] == "EUR"
)

bendra_eur_suma = sum(ivykdytu_eur_sumos, start=Decimal("0"))
print("Įvykdytų EUR operacijų suma:", bendra_eur_suma)
assert bendra_eur_suma == Decimal("435.65")


Įvykdytų EUR operacijų suma: 435.65


In [19]:
def filtruoti_pagal_rizika(irasai, slenkstis):
    """Pateikia įrašus po vieną; viso rezultato sąrašo nekuria."""
    for irasas in irasai:
        if irasas["rizikos_balas"] >= slenkstis:
            yield irasas

srautas = filtruoti_pagal_rizika(sandoriai, 0.80)
print("Objekto tipas:", type(srautas).__name__)
print("Pirmas rezultatas:", next(srautas)["id"])
print("Likę rezultatai:", [s["id"] for s in srautas])
print("Antrą kartą generatorius jau tuščias:", list(srautas))


Objekto tipas: generator
Pirmas rezultatas: TX-002
Likę rezultatai: ['TX-004']
Antrą kartą generatorius jau tuščias: []


> **Riba:** generatorius paprastai yra vienkartinis. Jei tuos pačius rezultatus reikia peržiūrėti kelis kartus, sukurkite generatorių iš naujo arba sąmoningai materializuokite `list(...)`. Srautinė forma taip pat nepašalina poreikio tikrinti duomenų kilmę ir klaidas.


---

## 8. `itertools`: sekų operacijos be rankinių indeksų

`itertools.pairwise()` pateikia gretimas poras. Kainų sekoje tai natūraliai atitinka ankstesnę ir dabartinę kainą. `accumulate()` kuria kaupiamą rezultatą.


In [20]:
from itertools import accumulate, pairwise
from operator import mul

paprastosios_grazos = [
    dabartine / ankstesne - 1
    for ankstesne, dabartine in pairwise(uzdarymo_kainos)
]

pradine_investicija = 1000.0
verte = list(
    accumulate(
        (1 + graza for graza in paprastosios_grazos),
        mul,
        initial=pradine_investicija,
    )
)

print("Grąžos:", [f"{g:.2%}" for g in paprastosios_grazos])
print("Vertės kelias:", [round(v, 2) for v in verte])
assert round(verte[-1], 2) == 1027.00


Grąžos: ['1.50%', '-0.69%', '2.38%', '-0.48%']
Vertės kelias: [1000.0, 1015.0, 1008.0, 1032.0, 1027.0]


**Kodėl čia paliktas `float`?** Rinkos grąžų analizėje slankiojo kablelio skaičiai dažnai yra priimtini aproksimaciniams skaičiavimams. Apskaitinėms pinigų sumoms toliau naudosime `Decimal`. Realioje strategijoje dar reikėtų dividendų, mokesčių, valiutos, prekybos kalendoriaus ir transakcijų sąnaudų.


---

## 9. `with`: ištekliaus gyvavimo ciklas matomas kode

Konteksto tvarkyklė užtikrina, kad failas, ryšys ar užraktas būtų tinkamai uždarytas net įvykus klaidai. Tai ne vien patogesnė sintaksė — tai patikimumo mechanizmas.

**FinTech pritaikymas:** audito žurnalo failai, CSV/JSON importas, duomenų bazės transakcijos, laikini tarpiniai failai.


In [22]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

santrauka = {
    "duomenu_data": "2026-09-15",
    "sandoriu_sk": len(sandoriai),
    "saltinis": "sintetinis mokomasis rinkinys",
}

with TemporaryDirectory() as laikinas_aplankas:
    kelias = Path(laikinas_aplankas) / "santrauka.json"
    with kelias.open("w", encoding="utf-8") as failas:
        json.dump(santrauka, failas, ensure_ascii=False, indent=2)
    with kelias.open("r", encoding="utf-8") as failas:
        nuskaityta = json.load(failas)

assert nuskaityta == santrauka
print(nuskaityta)


{'duomenu_data': '2026-09-15', 'sandoriu_sk': 5, 'saltinis': 'sintetinis mokomasis rinkinys'}


> **Produkcinė pastaba:** vien `with` neužtikrina atominio failo pakeitimo, prieigos kontrolės, šifravimo ar žurnalo nekintamumo. Šie reikalavimai priklauso nuo sistemos rizikos modelio.


---

## 10. `Decimal`: dešimtainė aritmetika pinigams

Dvejetainis `float` negali tiksliai atvaizduoti daugelio dešimtainių trupmenų. `Decimal`, sukurtas **iš teksto**, tiksliai atvaizduoja įvestą dešimtainį skaičių. Vis dėlto dalyba ir galutinis pinigų vieneto apvalinimas tebėra sąmoningi sprendimai.


In [23]:
from decimal import ROUND_HALF_EVEN, ROUND_HALF_UP

print("float 0.1 + 0.2 =", 0.1 + 0.2)
print("Decimal('0.1') + Decimal('0.2') =", Decimal("0.1") + Decimal("0.2"))

centas = Decimal("0.01")
tarpine_suma = Decimal("19.985")
bankinis = tarpine_suma.quantize(centas, rounding=ROUND_HALF_EVEN)
komercinis = tarpine_suma.quantize(centas, rounding=ROUND_HALF_UP)

print("ROUND_HALF_EVEN:", bankinis)
print("ROUND_HALF_UP:", komercinis)


float 0.1 + 0.2 = 0.30000000000000004
Decimal('0.1') + Decimal('0.2') = 0.3
ROUND_HALF_EVEN: 19.98
ROUND_HALF_UP: 19.99


In [24]:
vieneto_kaina = Decimal("12.49")
kiekis = 7
mokestis = Decimal("0.35")
galutine_suma = (vieneto_kaina * kiekis + mokestis).quantize(Decimal("0.01"))

print("Galutinė suma:", galutine_suma, "EUR")
assert galutine_suma == Decimal("87.78")


Galutinė suma: 87.78 EUR


> **Svarbu:** `Decimal(0.1)` paveldi jau atsiradusią `float` aproksimaciją. Naudokite `Decimal("0.1")` arba patikimai gautą tekstinę reikšmę. Apvalinimo taisyklę turi apibrėžti verslo, apskaitos ar reguliacinis reikalavimas — jos negalima pasirinkti vien pagal patogumą.


---

## 11. `dataclass`: domeno objektas be pasikartojančio techninio kodo

`@dataclass` sugeneruoja inicializavimą, aiškų atvaizdavimą ir palyginimą. `frozen=True` neleidžia netyčia pakeisti objekto laukų, o `slots=True` aiškiai apriboja jo atributus.

**FinTech pritaikymas:** pavedimas, kotiruotė, sąskaitos įrašas, rizikos sprendimas. Tai nėra duomenų bazė ar pilna validavimo sistema, bet geras mažo domeno modelio pagrindas.


In [25]:
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class Pavedimas:
    id: str
    suma: Decimal
    valiuta: str
    rizikos_balas: float

    def __post_init__(self):
        if self.suma <= 0:
            raise ValueError("Suma turi būti teigiama.")
        if not 0.0 <= self.rizikos_balas <= 1.0:
            raise ValueError("Rizikos balas turi būti intervale [0, 1].")
        if len(self.valiuta) != 3 or not self.valiuta.isupper():
            raise ValueError("Valiutai naudojamas 3 didžiųjų raidžių kodas.")

pavedimas = Pavedimas("PAY-101", Decimal("250.00"), "EUR", 0.18)
print(pavedimas)

try:
    Pavedimas("PAY-ERR", Decimal("-1.00"), "EUR", 0.20)
except ValueError as klaida:
    print("Validavimo klaida:", klaida)


Pavedimas(id='PAY-101', suma=Decimal('250.00'), valiuta='EUR', rizikos_balas=0.18)
Validavimo klaida: Suma turi būti teigiama.


> **Riba:** `frozen=True` nepadaro giliai nekintamų visų įdėtų objektų, o tipo anotacijos vykdymo metu savaime nevaliduojamos. Kritinėms riboms reikalingi testai ir aiškiai apibrėžta schema.


---

## 12. `datetime` + `zoneinfo`: laiko momentas, ne vien laikrodžio tekstas

Rinkos atsidaro skirtingose laiko juostose, o vasaros laikas keičia UTC poslinkį. „Naivus“ `datetime` be laiko juostos nepasako, kurio pasaulio taško laikas turėtas omenyje.


In [26]:
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

try:
    vilnius = ZoneInfo("Europe/Vilnius")
    niujorkas = ZoneInfo("America/New_York")

    ivykiu_laikas_vilniuje = datetime(2026, 9, 15, 16, 0, tzinfo=vilnius)
    tas_pats_momentas_niujorke = ivykiu_laikas_vilniuje.astimezone(niujorkas)

    print("Vilnius:", ivykiu_laikas_vilniuje.isoformat())
    print("Niujorkas:", tas_pats_momentas_niujorke.isoformat())
    assert ivykiu_laikas_vilniuje.timestamp() == tas_pats_momentas_niujorke.timestamp()
except ZoneInfoNotFoundError:
    print("Šioje sistemoje nėra IANA laiko juostų duomenų; žr. diegimo pastabą žemiau.")


Šioje sistemoje nėra IANA laiko juostų duomenų; žr. diegimo pastabą žemiau.


> **Diegimo pastaba:** `zoneinfo` yra standartinėje bibliotekoje, tačiau laiko juostų duomenis ima iš operacinės sistemos. Kai kurios aplinkos jų neturi ir tada diegimo apraše reikia numatyti IANA duomenų paketą. Produkcinėje sistemoje verta saugoti UTC momentą ir atskirai vartotojo ar rinkos laiko juostos identifikatorių.


---

## 13. `for ... else`: paieška baigėsi be `break`

`else` po ciklo įvykdomas tik tada, kai ciklas nebuvo nutrauktas su `break`. Konstrukcija glausta, bet ne visiems iš karto pažįstama, todėl tinka kaip papildomas, o ne pirmasis pasirinkimas.

**FinTech pritaikymas:** ieškoti pirmos neatitinkančios kontrolinės taisyklės; jei jos nėra, patvirtinti, kad visas rinkinys praėjo patikrą.


In [27]:
ieskomas_id = "TX-999"
for sandoris in sandoriai:
    if sandoris["id"] == ieskomas_id:
        print("Rastas:", sandoris)
        break
else:
    print(f"{ieskomas_id} nerastas — ciklas baigėsi be break.")


TX-999 nerastas — ciklas baigėsi be break.


> Jei komanda konstrukcijos nepažįsta, vardinė funkcija su ankstyvu `return` arba `next(..., None)` dažnai bus lengviau prižiūrima. „Pythoniška“ reiškia ir atsižvelgti į skaitytoją.


---

## 14. Kritinis skaitomumo pratimas: ne kiekviena viena eilutė yra gera

Toliau abi išraiškos grąžina tą patį rezultatą. Pirmoji sutalpina konvertavimą, filtrą ir verslo slenkstį vienoje eilutėje. Antroji suteikia taisyklei vardą, leidžia ją testuoti atskirai ir išlaiko trumpą atranką.


In [28]:
# Techniškai teisinga, bet sunkiau audituoti.
neaiskus_rezultatas = [s["id"] for s in sandoriai if s["busena"] == "ivykdytas" and Decimal(s["suma"]) >= Decimal("500") and s["rizikos_balas"] >= 0.8]

DIDELES_SUMOS_SLENKSTIS = Decimal("500")
AUKSTOS_RIZIKOS_SLENKSTIS = 0.80

def reikia_rankines_perziuros(sandoris):
    yra_ivykdytas = sandoris["busena"] == "ivykdytas"
    yra_didele_suma = Decimal(sandoris["suma"]) >= DIDELES_SUMOS_SLENKSTIS
    yra_didele_rizika = sandoris["rizikos_balas"] >= AUKSTOS_RIZIKOS_SLENKSTIS
    return yra_ivykdytas and yra_didele_suma and yra_didele_rizika

aiskus_rezultatas = [s["id"] for s in sandoriai if reikia_rankines_perziuros(s)]
assert neaiskus_rezultatas == aiskus_rezultatas
print(aiskus_rezultatas)


['TX-002', 'TX-004']


Klausimai diskusijai:

1. Kurioje versijoje lengviau pakeisti slenkstį ir paaiškinti jo prasmę?
2. Kurią versiją lengviau testuoti su ribinėmis reikšmėmis `499.99`, `500.00` ir `500.01`?
3. Ar funkcijos vardas pakankamai tikslus, jei taisyklė tik pažymi kandidatą, o ne priima galutinį sprendimą?


---

## 15. Integruotas mini atvejis: valiutinių pavedimų kontrolė

**Problema.** Turime kelių valiutų pavedimus. Reikia patikrinti duomenis, perskaičiuoti į EUR pagal užfiksuotą iliustracinę kursų lentelę, suskaičiuoti ekspoziciją ir sudaryti rankinės peržiūros eilę.

**Duomenų kilmė.** Visi įrašai sintetiniai. Kurso reikšmė reiškia, kiek EUR atitinka vienas valiutos vienetas; tai nėra dabartinis rinkos kursas.

Šiame sprendime sąmoningai sujungiami: `dataclass`, `Decimal`, `all`, generatorinė išraiška, `sorted(key=...)`, išpakavimas ir automatinės patikros.


In [29]:
pavedimai = [
    Pavedimas("PAY-201", Decimal("120.00"), "EUR", 0.15),
    Pavedimas("PAY-202", Decimal("850.00"), "USD", 0.88),
    Pavedimas("PAY-203", Decimal("600.00"), "GBP", 0.72),
    Pavedimas("PAY-204", Decimal("75.50"),  "EUR", 0.95),
]

# 1 valiutos vienetas = nurodytas EUR kiekis. Reikšmės tik mokomosios.
eur_kursai = {
    "EUR": Decimal("1.00"),
    "USD": Decimal("0.92"),
    "GBP": Decimal("1.17"),
}

assert all(p.valiuta in eur_kursai for p in pavedimai), "Trūksta valiutos kurso."

def suma_eur(pavedimas):
    return (pavedimas.suma * eur_kursai[pavedimas.valiuta]).quantize(Decimal("0.01"))

bendra_ekspozicija = sum((suma_eur(p) for p in pavedimai), start=Decimal("0"))
perziuros_eile = sorted(
    pavedimai,
    key=lambda p: (p.rizikos_balas, suma_eur(p)),
    reverse=True,
)

print("Bendra ekspozicija:", bendra_ekspozicija, "EUR")
print("Peržiūros eilė:")
for nr, pavedimas in enumerate(perziuros_eile, start=1):
    print(f"  {nr}. {pavedimas.id}: {suma_eur(pavedimas)} EUR; rizika={pavedimas.rizikos_balas:.2f}")


Bendra ekspozicija: 1679.50 EUR
Peržiūros eilė:
  1. PAY-204: 75.50 EUR; rizika=0.95
  2. PAY-202: 782.00 EUR; rizika=0.88
  3. PAY-203: 702.00 EUR; rizika=0.72
  4. PAY-201: 120.00 EUR; rizika=0.15


### Minimalūs automatiniai testai

Testai apima ne vien „laimingą kelią“: tikriname valiutos konvertavimą, bendrą sumą, prioriteto tvarką ir dvi netinkamas ribines reikšmes. Produkciniame projekte testai būtų atskirame testų modulyje.


In [30]:
assert suma_eur(pavedimai[1]) == Decimal("782.00")
assert bendra_ekspozicija == Decimal("1679.50")
assert [p.id for p in perziuros_eile] == ["PAY-204", "PAY-202", "PAY-203", "PAY-201"]

for netinkami_argumentai in [
    ("PAY-X", Decimal("0.00"), "EUR", 0.50),
    ("PAY-Y", Decimal("1.00"), "EUR", 1.01),
]:
    try:
        Pavedimas(*netinkami_argumentai)
    except ValueError:
        pass
    else:
        raise AssertionError("Tikėtasi ValueError")

print("Visi mini atvejo testai sėkmingi.")


Visi mini atvejo testai sėkmingi.


### Ribotumai ir plėtra

- Kursai realioje sistemoje turėtų turėti šaltinį, laiko žymą ir versiją.
- Reikia apibrėžti bazinę valiutą, kotiravimo kryptį, apvalinimo momentą ir taisyklę.
- Rizikos balas nėra sprendimas: reikia kalibruoto modelio, slenksčio pagrindimo, klaidingų teigiamų ir klaidingų neigiamų atvejų vertinimo.
- Didesniems srautams reikėtų patvarios saugyklos, idempotentiškumo, prieigos kontrolės ir audito pėdsako.
- Finansinių rezultatų vertinimui būtina atskirti mokymo ir tikrinimo laikotarpius bei vengti ateities informacijos.


---

## 16. Praktinė užduotis trimis lygiais

### A. Branduolys

Duotas naujas penkių pavedimų sąrašas. Parašykite kodą, kuris:

1. su `all()` patikrina, ar visoms valiutoms turime kursą;
2. generatorine išraiška apskaičiuoja bendrą EUR ekspoziciją;
3. su `enumerate()` išveda tris didžiausios rizikos pavedimus;
4. turi bent tris `assert` patikras, įskaitant ribinę reikšmę.

**Atramos:** remkitės `suma_eur()`, `sorted(..., key=...)` ir `Decimal` pavyzdžiais.

### B. Tyrimo dalis

Palyginkite dvi rankinės peržiūros taisykles:

- bazinė: `rizikos_balas >= 0.80`;
- alternatyvi: balo ir EUR sumos kombinacija, kurią suformuluojate patys.

Pasirinkite bent dvi metrikas (pavyzdžiui, pažymėtų atvejų skaičių ir bendrą pažymėtą ekspoziciją), paaiškinkite metodo pasirinkimą, atlikite jautrumo patikrą bent trims slenksčiams ir suformuluokite ribotumus. Sintetiniams duomenims nedarykite realaus pasaulio išvadų.

### C. Išplėtimas pažengusiems

Sukurkite tingią apdorojimo grandinę su generatoriais, kuri:

1. priima potencialiai didelį pavedimų srautą;
2. validavimo klaidas registruoja atskirai, o tinkamų įrašų srauto nenutraukia;
3. užtikrina deterministinę rezultatų tvarką;
4. įrašo JSON audito santrauką per `with`;
5. testais įrodo, kad rezultatas nepriklauso nuo duomenų skaidymo į porcijas.

Pažengusiųjų tikslas — ne daugiau eilučių, o gilesnis patikimumo ir srautinio vykdymo pagrindimas.


In [26]:
# Vieta studento bandymui. Ląstelė sąmoningai vykdoma be klaidų.
# TODO: įrašykite savo A, B arba C dalies sprendimą.
pass


## Pasitikrinimo klausimai

1. Kodėl `zip(strict=True)` finansinių duomenų derinime dažnai saugesnis už `zip()`?
2. Kuo `None`, nulis ir tuščias sąrašas skiriasi verslo prasme?
3. Kada generatorius taupo atmintį, o kada sukelia netikėtumą?
4. Kodėl piniginį `Decimal` verta kurti iš teksto?
5. Kokią papildomą informaciją turi laiko juostą žinantis `datetime`?
6. Kuo vardinė `key=` funkcija gali būti geresnė už `lambda`?
7. Kur integruotame pavyzdyje yra pedagoginis supaprastinimas, o kur — automatiškai patikrintas faktas?


## Generatyvinio DI naudojimo deklaracijos šablonas

> **Naudotas įrankis ir paskirtis:** …  
> **Priimti reikšmingi pasiūlymai:** …  
> **Atmesti arba pakeisti pasiūlymai ir kodėl:** …  
> **Kaip patikrinau kodą ir skaičiavimus:** …  
> **Ką galiu individualiai paaiškinti ar atkurti:** …

DI sugeneruotas kodas nėra patikra. Studentas lieka atsakingas už testus, prielaidas, duomenų kilmę ir išvadų pagrįstumą.


## Santrauka

Pythoniškumas nėra kuo trumpesnis kodas. Šioje užrašinėje jis reiškė:

- tiesiogiai išreikšti ketinimą (`for objektas in seka`, `any`, `all`, `key=`);
- anksti aptikti duomenų neatitikimus (`zip(strict=True)`, validacija, testai);
- pasirinkti domenui tinkamą abstrakciją (`Decimal`, `zoneinfo`, `dataclass`, `with`);
- atskirti srautinį skaičiavimą nuo materializuoto rezultato (generatoriai);
- palikti kode matomas finansines ir metodologines prielaidas.

### Oficialūs šaltiniai

- [Python vadovas: duomenų struktūros](https://docs.python.org/3/tutorial/datastructures.html)
- [Python vadovas: valdymo struktūros](https://docs.python.org/3/tutorial/controlflow.html)
- [Generatoriai](https://docs.python.org/3/tutorial/classes.html#generators)
- [`with` sakinys](https://docs.python.org/3/reference/compound_stmts.html#the-with-statement)
- [`decimal`](https://docs.python.org/3/library/decimal.html)
- [`itertools`](https://docs.python.org/3/library/itertools.html)
- [`dataclasses`](https://docs.python.org/3/library/dataclasses.html)
- [`zoneinfo`](https://docs.python.org/3/library/zoneinfo.html)

Užrašinė sukurta ir patikrinta 2026-09-15. Jos pavyzdžiai sąmoningai apsiriboja Python kalbos priemonėmis ir standartine biblioteka.
